# Exercise 3: prompt experiment – Flow and API

## Tasks

1. **Plan:** write two prompt variants that differ in one building block, and choose one attribute to score.
2. **Generate:** create 3 pictures per variant in Flow, everything else unchanged.
3. **Upload:** save the 6 pictures to Codespaces.
4. **Score:** let the model rate the attribute and check 2 pictures by hand.

## Setup

Select the **Python 3.11** kernel. The API key comes from your `OPENAI_API_KEY` Codespaces secret.

In [ ]:
import base64
import json
import mimetypes
from pathlib import Path

import numpy as np
import pandas as pd
from openai import OpenAI
from PIL import Image, ImageOps
from pydantic import BaseModel

client = OpenAI(max_retries=5, timeout=60)  # reads OPENAI_API_KEY, retries if the API is busy
MODEL = "gpt-6-luna"

DATA = Path("..") / "data"  # the course images and videos

## Helper functions

Two small functions do the API work in this notebook:

- `image_to_input(path)` turns an image file into text (base64), because the API request is sent as text.
- `ask(prompt, answer_format, images)` sends the prompt and the images, writes a log line and returns the answer as a Python dict.

In [ ]:
def image_to_input(path):
    """Turn an image file into the text format the API expects."""
    image_type = mimetypes.guess_type(path)[0]  # e.g. "image/jpeg"
    image_text = base64.b64encode(path.read_bytes()).decode()  # bytes -> text
    return {
        "type": "input_image",
        "image_url": f"data:{image_type};base64,{image_text}",
        "detail": "low",  # low detail = fewer tokens = cheaper
    }


def ask(prompt, answer_format, images):
    """Send a prompt and a list of images. Returns the answer as a dict."""
    # 1. Put the text and the images into one message.
    content = [{"type": "input_text", "text": prompt}]
    for path in images:
        content.append(image_to_input(path))

    # 2. Send the message. The answer must follow answer_format.
    response = client.responses.parse(
        model=MODEL,
        input=[{"role": "user", "content": content}],
        text_format=answer_format,
        reasoning={"effort": "none"},
        max_output_tokens=500,
        store=False,
    )

    # 3. Log prompt, model and raw answer.
    entry = {"prompt": prompt, "model": MODEL, "answer": response.output_text}
    with open("api_log.jsonl", "a", encoding="utf-8") as log:
        log.write(json.dumps(entry, ensure_ascii=False) + "\n")

    # 4. Return the answer as a Python dict.
    return response.output_parsed.model_dump()

## 1. Warm-up: adding noise (no API)

Diffusion training mixes an image with noise. The higher the level, the less of the original is left.

In [ ]:
image = Image.open(DATA / "images" / "lake.jpg").convert("RGB").resize((256, 256))
x0 = np.asarray(image) / 255  # pixels as numbers between 0 and 1
noise = np.random.default_rng(26).normal(size=x0.shape)

levels = [0, 0.25, 0.5, 0.75, 1]
noisy_images = []
for level in levels:
    mixed = np.sqrt(1 - level) * x0 + np.sqrt(level) * noise
    noisy_images.append((mixed.clip(0, 1) * 255).astype("uint8"))  # back to 0–255

print("Noise levels from left to right:", levels)
display(Image.fromarray(np.hstack(noisy_images)))  # all images side by side

## 2. Plan

Fill the building blocks. Only **one** block changes between the variants, the rest is fixed.

In [ ]:
TEMPLATE = "{subject}, {setting}, {light}, {composition}"
BASE = dict(
    subject="a red wooden rowboat",
    setting="on a mountain lake",
    light="at dawn, soft light",
)
COMPOSITION = {
    "a": "wide shot",
    "b": "wide shot, boat in the left third",
}

for variant in COMPOSITION:
    prompt = TEMPLATE.format(**BASE, composition=COMPOSITION[variant])
    print(variant, ":", prompt)

Decide **before** generating what you will score:

In [ ]:
ATTRIBUTE = "The center of the red rowboat is in the left third of the image."

## 3. Generate in Flow and upload

In Flow, generate **3 pictures per prompt** with the same model and aspect ratio (images are free). Upload the six files into the folder `flow_images/` next to this notebook and name them:

`variant_a_1`, `variant_a_2`, `variant_a_3`, `variant_b_1`, `variant_b_2`, `variant_b_3` (`.jpg` or `.png`)

In [ ]:
Path("flow_images").mkdir(exist_ok=True)
paths = sorted(Path("flow_images").glob("variant_*"))
print(len(paths), "images found (expected: 6)")

## 4. Score

The same yes/no question for all six pictures.

In [ ]:
class Rating(BaseModel):
    matches: bool
    reason: str


PROMPT = f"Is this true? {ATTRIBUTE} If there is no boat, answer false. Give a short reason."

In [ ]:
rows = []
for path in paths:
    row = ask(PROMPT, Rating, [path])
    row["file"] = path.name
    row["variant"] = path.name.split("_")[1]  # "variant_a_1.jpg" -> "a"
    rows.append(row)

ratings = pd.DataFrame(rows)
ratings

In [ ]:
# True counts as 1, False as 0 – so the mean is the share of "True".
ratings.groupby("variant")["matches"].mean()

## 5. Check by hand

Look at the first picture of each variant. Is the boat in the left third?

In [ ]:
print(paths[0].name, "and", paths[3].name)
display(ImageOps.contain(Image.open(paths[0]), (400, 300)))
display(ImageOps.contain(Image.open(paths[3]), (400, 300)))

In [ ]:
# Replace None with your own answer: True or False.
my_rating = {
    paths[0].name: None,
    paths[3].name: None,
}

for file in my_rating:
    model = ratings.loc[ratings["file"] == file, "matches"].iloc[0]
    print(file, "| model:", model, "| me:", my_rating[file])

**Check yourself:** Why does the same prompt give different pictures – and what does that mean for your experiment with only 3 pictures per variant?

### Jupyter notebook --footer info--

In [ ]:
import os
import platform
import socket
from platform import python_version
from datetime import datetime

print('-----------------------------------')
print(os.name.upper())
print(platform.system(), '|', platform.release())
print('Datetime:', datetime.now().strftime("%Y-%m-%d %H:%M:%S"))
print('Python Version:', python_version())
print('-----------------------------------')